# BED export

BED output is a pure adapter over candidate records. It never changes a
candidate's coordinates or score.

In [ ]:
#| default_exp bed

In [ ]:
#| export
from __future__ import annotations

from collections.abc import Iterable

from probeharbour.models import Candidate

In [ ]:
#| export
def candidates_to_bed(candidates: Iterable[Candidate], *, include_rgb: bool = True) -> str:
    """Render candidates with coordinates as BED6 or BED12 text."""
    lines: list[str] = []
    for candidate in candidates:
        coords = candidate.coordinates
        if coords is None:
            continue
        score = int(max(0, min(1000, round((candidate.stored_raw_design_score or 0.0) * 1000))))
        rgb = "85,168,104" if include_rgb else "0,0,0"
        arms = (coords.rbd5_start, coords.rbd5_end, coords.rbd3_start, coords.rbd3_end)
        if all(value is not None for value in arms):
            b5_size, b3_size = coords.rbd5_end - coords.rbd5_start, coords.rbd3_end - coords.rbd3_start
            b5_rel, b3_rel = coords.rbd5_start - coords.start, coords.rbd3_start - coords.start
            lines.append(
                f"{coords.chrom}\t{coords.start}\t{coords.end}\t{candidate.candidate_id}\t{score}\t{coords.strand}\t"
                f"{coords.start}\t{coords.end}\t{rgb}\t2\t{b5_size},{b3_size}\t{b5_rel},{b3_rel}"
            )
        else:
            lines.append(f"{coords.chrom}\t{coords.start}\t{coords.end}\t{candidate.candidate_id}\t{score}\t{coords.strand}")
    return "\n".join(lines) + ("\n" if lines else "")